In [3]:
!pip install -q transformers accelerate

In [4]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

login(userdata.get("HF_TOKEN"))

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

def generate_text(prompt, temperature=0.7, max_tokens=3000):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
    )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [6]:
import csv
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, "/content")

from prompting import build_writing_prompt

TEMPERATURE = 0.7
MAX_TOKENS = 3000
TARGET_WORDS = 400
MODEL_NAME = "Llama 3.2 3B"

STRATEGIES = ["Zero-shot", "Few-shot", "Structured"]
COLUMNS = ["prompt_id", "model", "strategy", "story", "words", "seconds"]

prompts_df = pd.read_csv("/content/writingprompts_sample.csv")
examples = pd.read_csv("/content/fewshot_examples.csv").to_dict("records")
output_path = Path("/content/stories_llama.csv")

done = set()
if output_path.exists():
    existing = pd.read_csv(output_path)
    done = set(zip(existing["prompt_id"], existing["model"], existing["strategy"]))
else:
    with open(output_path, "w", newline="", encoding="utf-8") as file:
        csv.writer(file).writerow(COLUMNS)

total = len(prompts_df) * len(STRATEGIES)
counter = 0

for _, row in prompts_df.iterrows():
    for strategy in STRATEGIES:
        counter += 1

        if (row["prompt_id"], MODEL_NAME, strategy) in done:
            continue

        prompt = build_writing_prompt(
            row["prompt"],
            strategy,
            examples=examples,
            target_words=TARGET_WORDS,
        )

        start = time.time()
        try:
            story = generate_text(prompt, temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
        except Exception as error:
            print(f"[{counter}/{total}] FAILED {strategy} | prompt {row['prompt_id']}: {error}")
            continue

        seconds = round(time.time() - start, 1)

        with open(output_path, "a", newline="", encoding="utf-8") as file:
            csv.writer(file).writerow([
                row["prompt_id"],
                MODEL_NAME,
                strategy,
                story,
                len(story.split()),
                seconds,
            ])

        print(f"[{counter}/{total}] OK {strategy} | prompt {row['prompt_id']} | {len(story.split())} words | {seconds}s")

print("Done. Saved to:", output_path)

[6/300] OK Structured | prompt 2 | 352 words | 17.7s
[7/300] OK Zero-shot | prompt 3 | 425 words | 19.1s
[8/300] OK Few-shot | prompt 3 | 357 words | 18.8s
[9/300] OK Structured | prompt 3 | 376 words | 17.3s
[10/300] OK Zero-shot | prompt 4 | 376 words | 17.2s
[11/300] OK Few-shot | prompt 4 | 517 words | 26.5s
[12/300] OK Structured | prompt 4 | 475 words | 21.4s
[13/300] OK Zero-shot | prompt 5 | 418 words | 19.9s
[14/300] OK Few-shot | prompt 5 | 389 words | 19.5s
[15/300] OK Structured | prompt 5 | 416 words | 19.2s
[16/300] OK Zero-shot | prompt 6 | 398 words | 17.5s
[17/300] OK Few-shot | prompt 6 | 505 words | 26.0s
[18/300] OK Structured | prompt 6 | 452 words | 20.2s
[19/300] OK Zero-shot | prompt 7 | 397 words | 17.5s
[20/300] OK Few-shot | prompt 7 | 413 words | 20.3s
[21/300] OK Structured | prompt 7 | 422 words | 18.5s
[22/300] OK Zero-shot | prompt 8 | 450 words | 20.9s
[23/300] OK Few-shot | prompt 8 | 602 words | 30.1s
[24/300] OK Structured | prompt 8 | 417 words | 18